# Notebook 03_05 — Feature Engineering: Statistical Feature Generation

Domain-knowledge feature engineering: derive summary statistics (mean, std, min, max,
range, median, skewness, kurtosis, 25th/75th percentile) across the original 20 raw
features for every row, then keep the top-K most variable statistics (selected on
training data only).

All operations are vectorized numpy/scipy calls — cheap even on 3.4M rows, no subsampling needed.

**Results saved incrementally to** `results/fe_statfeatures_raw.csv` — resumable on crash.

In [ ]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *
from scipy.stats import skew, kurtosis

print('Imports OK')

In [ ]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']

METHOD_NAME = 'StatFeatures'
RESULTS_CSV = f'{RESULTS_DIR}fe_statfeatures_raw.csv'

In [ ]:
def stat_transform(X):
    feats = [
        X.mean(axis=1, keepdims=True),
        X.std(axis=1, keepdims=True),
        X.min(axis=1, keepdims=True),
        X.max(axis=1, keepdims=True),
        (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1),
        np.median(X, axis=1, keepdims=True),
        skew(X, axis=1).reshape(-1, 1),
        kurtosis(X, axis=1).reshape(-1, 1),
        np.percentile(X, 25, axis=1).reshape(-1, 1),
        np.percentile(X, 75, axis=1).reshape(-1, 1),
    ]
    return np.nan_to_num(np.hstack(feats))


# Precompute stat features once (independent of K) -- selection of top-K happens per call
print('Computing statistical features for train/val/test ...')
t0 = time.time()
X_train_stat = stat_transform(X_train)
X_val_stat   = stat_transform(X_val)
X_test_stat  = stat_transform(X_test)
print(f'Done in {time.time()-t0:.1f}s  |  shape: {X_train_stat.shape}')

In [ ]:
N_STAT_FEATURES = X_train_stat.shape[1]   # only 10 statistics generated above
print(f'Available statistical features: {N_STAT_FEATURES}  (K > {N_STAT_FEATURES} will be clipped)')


def transform_fn(K):
    # Select top-K statistical features by training-set variance.
    # Only N_STAT_FEATURES (=10) statistics exist; K=16 is clipped and zero-padded,
    # exactly like the LDA n_classes-1 constraint in 03_02.
    K_actual = min(K, N_STAT_FEATURES)
    var = X_train_stat.var(axis=0)
    top_K = np.argsort(var)[::-1][:K_actual]

    Xtr, Xva, Xte = X_train_stat[:, top_K], X_val_stat[:, top_K], X_test_stat[:, top_K]
    if K_actual < K:
        pad = K - K_actual
        Xtr = np.hstack([Xtr, np.zeros((len(Xtr), pad))])
        Xva = np.hstack([Xva, np.zeros((len(Xva), pad))])
        Xte = np.hstack([Xte, np.zeros((len(Xte), pad))])
        print(f'  [StatFeatures K={K}] clipped to {K_actual} available statistics, zero-padded to {K}')

    return Xtr, Xva, Xte, K_actual

In [ ]:
run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

In [ ]:
res_df = pd.read_csv(RESULTS_CSV)
print(res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std']).to_string())